In [ ]:
pip install mnist1d

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from mnist1d.data import make_dataset, get_dataset_args
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torchvision import transforms, datasets

## How to create 1D CNN for training and testing on 1D MNIST:

* Preparing our dataset
* Designing 1D CNN
* Train and test loop
* Visualizing our loss and accuracy

\\[ (N, C, L) \\]

Where:

- \\(N\\): Number of samples in each batch
- \\(C\\): Number of input channels
- \\(L\\): Length of each signal


In [ ]:
args = get_dataset_args()
data = make_dataset(args)

In [ ]:
data

{'x': array([[-0.33200567, -0.47191037, -0.77869708, ..., -0.7167305 ,
         -0.96195825, -0.89016639],
        [ 1.96548088,  2.30021254,  2.21339278, ..., -1.68751297,
         -1.42642212, -1.36421511],
        [-0.01930303,  0.07863308,  0.62488909, ..., -1.08555748,
         -1.79809121, -2.3153282 ],
        ...,
        [-1.69449348, -1.2805858 , -0.65125255, ..., -0.72365942,
         -0.38266551, -0.03165042],
        [ 0.99646721,  0.80844603,  0.1553411 , ..., -0.30286571,
         -0.28555128, -0.1183879 ],
        [-0.6939948 , -0.27532169,  0.29267968, ..., -1.21624513,
         -1.4649417 , -1.65099286]]),
 'x_test': array([[-0.07877078, -0.21073171, -0.09175138, ...,  1.13416264,
          0.84632982,  0.34613289],
        [ 0.47104725,  0.73107223,  0.78374075, ..., -0.46250157,
         -0.18451044,  0.09676974],
        [ 1.7738889 ,  1.26898205,  1.37670332, ..., -0.70019867,
         -0.10291667,  0.07403612],
        ...,
        [-0.6091856 , -0.11078297,  0.2

In [ ]:
print(type(data))
print(data.keys() if isinstance(data, dict) else data)

<class 'dict'>
dict_keys(['x', 'x_test', 'y', 'y_test', 't', 'templates'])


In [ ]:
Xtrain = data['x']
ytrain = data['y']

Xtest = data['x_test']
ytest = data['y_test']

In [ ]:
type(Xtrain), Xtrain.dtype, ytrain.dtype

(numpy.ndarray, dtype('float64'), dtype('int64'))

In [ ]:
X_train = torch.tensor(Xtrain, dtype= torch.float32)
y_train = torch.tensor(ytrain, dtype= torch.long)

X_test = torch.tensor(Xtest, dtype= torch.float32)
y_test = torch.tensor(ytest, dtype= torch.long)

In [ ]:
X_train.shape, y_train.shape

(torch.Size([4000, 40]), torch.Size([4000]))

`unsqueeze(1)` adds a new dimension at index 1, changing the shape from `(4000, 40)` to `(4000, 1, 40)` to match the required `(N, C, L)` input format of `Conv1d`.

In [ ]:
X_train = X_train.unsqueeze(1)
X_test = X_test.unsqueeze(1)

In [ ]:
X_train.shape

torch.Size([4000, 1, 40])

`TensorDataset` pairs each input sample with its corresponding label to create the training and test datasets.

In [ ]:
Tensortrain = TensorDataset(X_train, y_train)
Tensortest = TensorDataset(X_test, y_test)

`DataLoader` divides the dataset into batches of 64 samples and optionally shuffles their order during each iteration.

In [ ]:
Trainloader = DataLoader(Tensortrain, batch_size= 64, shuffle= True)
Traintest = DataLoader(Tensortest, batch_size= 64, shuffle= True)

# Designing 1D CNN

### Shape Changes Through the 1D CNN

The input tensor has shape **`(N, C, L)`**, where:
- `N`: batch size
- `C`: number of channels
- `L`: sequence length

For an input shape of `(64, 1, 40)`, the dimensions change as follows:

| Layer | Output Shape `(N, C, L)` | Explanation |
|---|---|---|
| Input | `(64, 1, 40)` | 1 input channel and sequence length 40 |
| `Conv1d(1, 16, kernel_size=3, padding=1, stride=1)` | `(64, 16, 40)` | Channels increase from 1 to 16; length stays the same |
| `BatchNorm1d(16)` | `(64, 16, 40)` | Shape remains unchanged |
| `ReLU()` | `(64, 16, 40)` | Shape remains unchanged |
| `MaxPool1d(kernel_size=2, stride=2)` | `(64, 16, 20)` | Sequence length is halved |
| `Conv1d(16, 32, kernel_size=3, padding=1, stride=1)` | `(64, 32, 20)` | Channels increase from 16 to 32 |
| `BatchNorm1d(32)` | `(64, 32, 20)` | Shape remains unchanged |
| `ReLU()` | `(64, 32, 20)` | Shape remains unchanged |
| `MaxPool1d(kernel_size=2, stride=2)` | `(64, 32, 10)` | Sequence length is halved |
| `Conv1d(32, 64, kernel_size=3, padding=1, stride=1)` | `(64, 64, 10)` | Channels increase from 32 to 64 |
| `BatchNorm1d(64)` | `(64, 64, 10)` | Shape remains unchanged |
| `ReLU()` | `(64, 64, 10)` | Shape remains unchanged |
| `AdaptiveAvgPool1d(1)` | `(64, 64, 1)` | Each channel is reduced to a single value |

**Key idea:** Convolutional layers change the number of channels, pooling layers reduce the sequence length, and Batch Normalization and ReLU preserve the tensor shape.

In [ ]:
class CNN1D(nn.Module):
  def __init__(self, in_channel, num_classes):

    super(CNN1D, self).__init__()
    # or `super().__init__()`

    self.Blocks = nn.Sequential(nn.Conv1d(in_channels= 1, out_channels= 16, kernel_size= 3, padding= 1, stride= 1),
                                nn.BatchNorm1d(num_features= 16),
                                nn.ReLu(),
                                nn.MaxPool1d(kernel_size= 2, stride= 2),


                                nn.Conv1d(in_channels= 16, out_channels= 32, kernel_size= 3, padding= 1, stride= 1),
                                nn.BatchNorm1d(num_features= 32),
                                nn.ReLU(),
                                nn.MaxPool1d(kernel_size= 2, stride= 2),


                                nn.Conv1d(in_channels= 32, out_channels= 64, kernel_size= 3, padding= 1, stride= 1),
                                nn.BatchNorm1d(num_features= 64),
                                nn.ReLU(),
                                nn.AdaptiveAvgPool1d(output_size= 1))